# RiskRadar AI: SEC Filing RAG System for Financial Risk Intelligence

## Project Introduction

RiskRadar AI is a Retrieval-Augmented Generation system that analyzes real SEC company filings and answers business questions with grounded citations.

The goal is to help users ask questions about company risks, strategy, competition, regulation, cybersecurity, financial pressure, and market uncertainty using real public company documents instead of relying on unsupported chatbot answers.

## 1. Business Problem

Public companies publish long SEC filings such as 10-K, 10-Q, and 8-K reports. These documents contain valuable information about company risks, financial condition, business strategy, competition, regulation, and future uncertainty.

However, these filings are often long, dense, and difficult to analyze manually.

A business analyst, investor, consultant, recruiter, or strategy team may want to quickly answer questions such as:

- What are NVIDIA's top AI-related risks?
- How does Tesla describe supply chain risk?
- What does Microsoft say about cybersecurity?
- How do AMD and NVIDIA compare on competition risk?
- What financial risks does JPMorgan mention?

The business problem is:

**How can we build a system that lets users ask natural language questions over real SEC filings and receive grounded answers with evidence and citations?**

## 2. Why This Project Matters

This project matters because it combines multiple real-world skills:

- Data ingestion from public APIs
- Text cleaning and document parsing
- Natural Language Processing
- Retrieval-Augmented Generation
- Vector databases
- Hybrid search
- Reranking
- Citation-grounded answer generation
- Evaluation
- API deployment
- Business-focused user interface

This is stronger than a basic chatbot because the system does not only generate text. It retrieves evidence from real company documents first, then uses that evidence to answer the user's question.

## 3. What Is Retrieval-Augmented Generation?

A normal chatbot works like this:

```text
User Question
→ Language Model
→ Answer

## 3. What Is Retrieval-Augmented Generation?

A normal chatbot works like this:

```text
User Question
→ Language Model
→ Answer

A RAG system works like this:

User Question
→ Search relevant documents
→ Retrieve supporting evidence
→ Send evidence to language model
→ Generate grounded answer with citations

RAG helps reduce hallucinations because the model is not answering from memory alone. It answers using retrieved context from real documents.

## 4. Project Workflow

The full project will follow this workflow:

```text
Business problem
→ Data sources
→ Data ingestion
→ Data understanding
→ Exploratory data analysis
→ Text cleaning
→ Section extraction
→ Chunking
→ Embeddings
→ Vector database
→ Baseline retrieval
→ Hybrid retrieval
→ Reranking
→ RAG answer generation
→ Citation grounding
→ RAG evaluation
→ FastAPI backend
→ Streamlit app
→ Monitoring
→ Business impact

## 5. System Flowchart

```text
User asks question
        ↓
Question preprocessing
        ↓
Company / year / filing filter
        ↓
Retriever searches SEC filing chunks
        ↓
Top document chunks are returned
        ↓
Reranker improves evidence quality
        ↓
LLM receives question + evidence
        ↓
Answer is generated
        ↓
Citations are attached
        ↓
Final answer is shown to the user
```

This is the simple version. Later, the project will include a stronger visual architecture diagram for the README.

## 6. Data Sources

The main data source will be SEC EDGAR.

We will use:

| Data Source | Data Type | Purpose |
|---|---|---|
| SEC Company Submissions API | JSON metadata | Find company filings |
| SEC 10-K filings | Long text documents | Annual business and risk analysis |
| SEC 10-Q filings | Long text documents | Quarterly updates |
| SEC 8-K filings | Long text documents | Major company events |
| SEC Company Facts API | Structured financial data | Revenue, assets, liabilities, and other financial metrics |

The first version of the project will focus on 10-K filings because they contain the richest risk and business information.

## 7. Target Users

This system could help:

- Data scientists building GenAI products
- Financial analysts reviewing companies
- Consultants comparing competitors
- Recruiters evaluating AI engineering skill
- Business teams researching risk exposure
- Startup founders researching market threats

## 8. Success Criteria

The project will be considered successful if it can:

1. Download real SEC filings.
2. Extract useful sections such as Risk Factors and Management Discussion.
3. Split long documents into searchable chunks.
4. Store chunks in a vector database.
5. Retrieve relevant evidence for a user question.
6. Generate answers using only retrieved context.
7. Return citations showing where the answer came from.
8. Evaluate retrieval and answer quality.
9. Deploy the system through an API or app.

## 9. Tools

The project will use:

| Tool | Purpose |
|---|---|
| Python | Main programming language |
| pandas | Data manipulation |
| requests | SEC API calls |
| BeautifulSoup | HTML and text cleaning |
| Chroma | Vector database |
| sentence-transformers | Embeddings |
| rank-bm25 | Keyword search |
| FastAPI | Backend API |
| Streamlit | User interface |
| DuckDB or SQLite | Logging and lightweight storage |
| Docker | Deployment packaging |

In [8]:
import sys
import pandas as pd
import requests
import bs4
import sklearn

print("Python version:", sys.version)
print("pandas version:", pd.__version__)
print("requests imported successfully")
print("BeautifulSoup imported successfully")
print("scikit-learn imported successfully")

Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
pandas version: 2.3.3
requests imported successfully
BeautifulSoup imported successfully
scikit-learn imported successfully


In [9]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
VECTORSTORE_DIR = DATA_DIR / "vectorstore"
REPORTS_DIR = PROJECT_ROOT / "reports"

paths = {
    "project_root": PROJECT_ROOT,
    "data": DATA_DIR,
    "raw": RAW_DIR,
    "processed": PROCESSED_DIR,
    "vectorstore": VECTORSTORE_DIR,
    "reports": REPORTS_DIR,
}

for name, path in paths.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"{name}: {path}")

project_root: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI
data: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data
raw: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\raw
processed: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed
vectorstore: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\vectorstore
reports: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\reports


In [10]:
companies = pd.DataFrame({
    "ticker": [
        "NVDA", 
        "MSFT", 
        "AAPL", 
        "TSLA", 
        "AMZN", 
        "META", 
        "AMD", 
        "PLTR", 
        "JPM", 
        "UNH"
    ],
    "company_name": [
        "NVIDIA",
        "Microsoft",
        "Apple",
        "Tesla",
        "Amazon",
        "Meta Platforms",
        "Advanced Micro Devices",
        "Palantir",
        "JPMorgan Chase",
        "UnitedHealth Group"
    ],
    "sector": [
        "Semiconductors",
        "Cloud / Software",
        "Consumer Technology",
        "Electric Vehicles",
        "E-Commerce / Cloud",
        "Social Media / AI",
        "Semiconductors",
        "Data / AI Software",
        "Banking",
        "Healthcare"
    ]
})

companies

,ticker,company_name,sector
0,NVDA,NVIDIA,Semiconductors
1,MSFT,Microsoft,Cloud / Software
2,AAPL,Apple,Consumer Technology
3,TSLA,Tesla,Electric Vehicles
4,AMZN,Amazon,E-Commerce / Cloud
5,META,Meta Platforms,Social Media / AI
6,AMD,Advanced Micro Devices,Semiconductors
7,PLTR,Palantir,Data / AI Software
8,JPM,JPMorgan Chase,Banking
9,UNH,UnitedHealth Group,Healthcare


In [11]:
company_file = PROCESSED_DIR / "company_universe.csv"

companies.to_csv(company_file, index=False)

print(f"Saved company universe to: {company_file}")

Saved company universe to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\company_universe.csv


#### SEC Request Header

In [12]:
SEC_HEADERS = {
    "User-Agent": "RiskRadarAI/0.1 tevinswright@gmail.com"
}

SEC_HEADERS

{'User-Agent': 'RiskRadarAI/0.1 tevinswright@gmail.com'}

In [13]:
ticker_url = "https://www.sec.gov/files/company_tickers.json"

response = requests.get(ticker_url, headers=SEC_HEADERS)

print("Status code:", response.status_code)
print("Response type:", response.headers.get("content-type"))

Status code: 200
Response type: application/json


In [14]:
ticker_data = response.json()

sec_tickers = pd.DataFrame.from_dict(ticker_data, orient="index")

sec_tickers.head()

,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,1652044,GOOGL,Alphabet Inc.
2,320193,AAPL,Apple Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC


#### Clean SEC Ticker Data

In [15]:
sec_tickers = sec_tickers.rename(columns={
    "cik_str": "cik",
    "ticker": "ticker",
    "title": "company_title"
})

sec_tickers["ticker"] = sec_tickers["ticker"].str.upper()
sec_tickers["cik_padded"] = sec_tickers["cik"].astype(str).str.zfill(10)

sec_tickers.head()

,cik,ticker,company_title,cik_padded
0,1045810,NVDA,NVIDIA CORP,0001045810
1,1652044,GOOGL,Alphabet Inc.,0001652044
2,320193,AAPL,Apple Inc.,0000320193
3,789019,MSFT,MICROSOFT CORP,0000789019
4,1018724,AMZN,AMAZON COM INC,0001018724


#### Match Our Companies to SEC CIKs

In [16]:
company_universe = companies.merge(
    sec_tickers[["ticker", "cik", "cik_padded", "company_title"]],
    on="ticker",
    how="left"
)

company_universe

,ticker,company_name,sector,cik,cik_padded,company_title
0,NVDA,NVIDIA,Semiconductors,1045810,0001045810,NVIDIA CORP
1,MSFT,Microsoft,Cloud / Software,789019,0000789019,MICROSOFT CORP
2,AAPL,Apple,Consumer Technology,320193,0000320193,Apple Inc.
3,TSLA,Tesla,Electric Vehicles,1318605,0001318605,"Tesla, Inc."
4,AMZN,Amazon,E-Commerce / Cloud,1018724,0001018724,AMAZON COM INC
5,META,Meta Platforms,Social Media / AI,1326801,0001326801,"Meta Platforms, Inc."
6,AMD,Advanced Micro Devices,Semiconductors,2488,0000002488,ADVANCED MICRO DEVICES INC
7,PLTR,Palantir,Data / AI Software,1321655,0001321655,Palantir Technologies Inc.
8,JPM,JPMorgan Chase,Banking,19617,0000019617,JPMORGAN CHASE & CO
9,UNH,UnitedHealth Group,Healthcare,731766,0000731766,UNITEDHEALTH GROUP INC


#### Save Company Universe With CIKs

In [17]:
company_cik_file = PROCESSED_DIR / "company_universe_with_cik.csv"

company_universe.to_csv(company_cik_file, index=False)

print(f"Saved company CIK file to: {company_cik_file}")

Saved company CIK file to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\data\processed\company_universe_with_cik.csv


#### Mini Project Summary Table

In [18]:
project_summary = pd.DataFrame({
    "component": [
        "Data source",
        "Document type",
        "Main technique",
        "Retriever",
        "Answer style",
        "Deployment goal"
    ],
    "choice": [
        "SEC EDGAR",
        "10-K annual filings",
        "Retrieval-Augmented Generation",
        "Vector search first, hybrid search later",
        "Grounded answers with citations",
        "FastAPI + Streamlit"
    ],
    "why_it_matters": [
        "Real public company data",
        "Rich risk and business information",
        "Reduces unsupported LLM answers",
        "Finds relevant filing evidence",
        "Makes answers auditable",
        "Shows production thinking"
    ]
})

project_summary

,component,choice,why_it_matters
0,Data source,SEC EDGAR,Real public company data
1,Document type,10-K annual filings,Rich risk and business information
2,Main technique,Retrieval-Augmented Generation,Reduces unsupported LLM answers
3,Retriever,"Vector search first, hybrid search later",Finds relevant filing evidence
4,Answer style,Grounded answers with citations,Makes answers auditable
5,Deployment goal,FastAPI + Streamlit,Shows production thinking


#### Write a Small Project Overview File

In [19]:
overview_text = """
# RiskRadar AI

RiskRadar AI is a Retrieval-Augmented Generation system that analyzes real SEC filings.

The system will allow users to ask business and financial risk questions about public companies and receive grounded answers with citations.

Core workflow:

Business question
→ SEC filing ingestion
→ section extraction
→ chunking
→ embeddings
→ vector search
→ RAG answer generation
→ citations
→ evaluation
→ deployment

Initial companies:
- NVIDIA
- Microsoft
- Apple
- Tesla
- Amazon
- Meta
- AMD
- Palantir
- JPMorgan Chase
- UnitedHealth Group
"""

overview_file = REPORTS_DIR / "project_overview.md"

overview_file.write_text(overview_text, encoding="utf-8")

print(f"Saved overview file to: {overview_file}")

Saved overview file to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\RiskRadar-AI\reports\project_overview.md


## 10. Conclusion

This notebook introduced the RiskRadar AI project.

The project will build a real-world RAG system that uses SEC filings as trusted source documents. The system will retrieve relevant filing sections, generate grounded answers, and provide citations so users can verify the response.

The next notebook will focus on data ingestion:

```text
Ticker
→ CIK lookup
→ SEC submissions API
→ filing metadata
→ 10-K selection
→ raw filing download
→ local storage
```